# Identifying Splice-Variants and Post-Prenylation Processing Variants in MaxQuant Results

# Setup

## Import and install Packages

In [ ]:
import sys
import os
import session_info

# Add the '0_functions' folder to sys.path
sys.path.append(os.path.join(os.getcwd(), '..', '00_functions'))

In [ ]:
import pandas as pd
import re
import openpyxl
from functions import read_fastafile

In [ ]:
# Display session information
session_info.show()

## Set working directory

In [ ]:
cwd = os.getcwd()
if not cwd.endswith("Isoform_Database/Jupyter_environment/Isoform_Database_SIAF/05_MaxQuant"):
    print("WARNING: The working directory is not set to the '05_MaxQuant'!")
    print("Current working directory:", cwd)
else:
    print(f"Working directory is correctly set to the '05_MaxQuant' folder (\"{cwd}\").")

In [ ]:
# Data directories
prenylation_dir = "../04_Prenylation_Database/data/"
mapping_dir = "../01_UniProt/data/mapping/"
isoform_database_dir = "../02_Isoform_Database/"

total_extracts_dir = "data/total_extracts/"
prenylation_enrichment_dir = "data/prenylation_enrichment/"

## Read in MaxQuant Results and Lists of isoforms and prenylated proteins

In [ ]:
mq_results_1 = pd.read_excel('data/proteinGroup_searchdb_1.xlsx') # used unique isoforms but not unique canonical

#total extracts
mq_results_stat_te = pd.read_excel(os.path.join(total_extracts_dir, 'proteinGroups_stat_te_nrdbln.xlsx')) # used unique isoforms and unique canonical
mq_results_act_te = pd.read_excel(os.path.join(total_extracts_dir, 'proteinGroups_act_te_nrdbln.xlsx')) # Th1-Cells activated vs non-activated
mq_results_FTI_te = pd.read_excel(os.path.join(total_extracts_dir, 'proteinGroups_FTI_te_nrdbln.xlsx')) # Th1-Cells activated +/- treatment with farnesyl transferase inhibitor

#prenylation enriched
mq_results_act_clickit = pd.read_excel(os.path.join(prenylation_enrichment_dir, 'protein_Groups_act_clickit_nrdbln.xlsx')) # Prenylation enrichment samples (with click-it reaction), Th1-Cells activated vs non-activated

isoform_fasta = read_fastafile(os.path.join(isoform_database_dir, 'TryP_Isoform_Database_Final.fasta'))
prenylation_fasta = read_fastafile(os.path.join(prenylation_dir, 'Post-Prenylation_Processing_Database.fasta'))

# Filter Fasta headers by Isoform and Prenylation fasta headers

In [ ]:
isoform_set = set(isoform_fasta['seqID'])
prenylation_set = set(prenylation_fasta['seqID'])

In [ ]:
def filter_maxquant_variants(df, isoform_set, prenylation_set, id_col='Protein IDs', header_col='Fasta headers'):
    """
    Cleans MaxQuant results and splits them into Isoforms and Prenylation variants.
    """
    # 1. Clean Decoys and Contaminants first
    # This ensures your statistical FDR is respected
    initial_count = len(df)
    clean_df = df[
        ~df[id_col].str.contains(r'REV_|CON_', case=False, na=False)
    ].copy()
    
    # Filter by MaxQuant's internal flag columns if they exist
    for col in ['Reverse', 'Potential contaminant']:
        if col in clean_df.columns:
            clean_df = clean_df[clean_df[col] != '+']

    # 2. Helper function for set intersection
    def check_match(header_str, target_set):
        if pd.isna(header_str): return False
        return any(h.strip() in target_set for h in str(header_str).split(';'))

    # 3. Create the two filtered DataFrames
    df_isoforms = clean_df[
        clean_df[header_col].apply(lambda x: check_match(x, isoform_set))
    ].copy()
    
    df_prenylation = clean_df[
        clean_df[header_col].apply(lambda x: check_match(x, prenylation_set))
    ].copy()

    # Reporting
    print(f"Summary:")
    print(f"- Decoys/Contaminants removed: {initial_count - len(clean_df)}")
    print(f"- Isoform hits: {len(df_isoforms)}")
    print(f"- Prenylation hits: {len(df_prenylation)}")
    
    return df_isoforms, df_prenylation

In [ ]:
# Execution:
isoforms_df_1, prenylation_df_1 = filter_maxquant_variants(mq_results_1, isoform_set, prenylation_set)

#total extracts
isoforms_stat_te, prenylation_stat_te = filter_maxquant_variants(mq_results_stat_te, isoform_set, prenylation_set)
isoforms_act_te, prenylation_act_te = filter_maxquant_variants(mq_results_act_te, isoform_set, prenylation_set)
isoforms_FTI_te, prenylation_FTI_te = filter_maxquant_variants(mq_results_FTI_te, isoform_set, prenylation_set)

#prentylation enrichment
isoforms_act_clickit, prenylation_act_clickit = filter_maxquant_variants(mq_results_act_clickit, isoform_set, prenylation_set)

In [ ]:
# Save splice variants as csv 
isoforms_df_1.to_csv('data/MQ_splice_variants_1.csv', index=False)

isoforms_stat_te.to_csv(os.path.join(total_extracts_dir, 'MQ_splice_variants_stat_te.csv'), index=False)
isoforms_act_te.to_csv(os.path.join(total_extracts_dir, 'MQ_splice_variants_act_te.csv'), index=False)
isoforms_FTI_te.to_csv(os.path.join(total_extracts_dir, 'MQ_splice_variants_FTI_te.csv'), index=False)

isoforms_act_clickit.to_csv(os.path.join(prenylation_enrichment_dir, 'MQ_splice_variants_act_clickit.csv'), index=False)

In [ ]:
# Save post-prenylation processing variants as csv 
prenylation_df_1.to_csv('data/MQ_prenylation_variants_1.csv', index=False)

prenylation_stat_te.to_csv(os.path.join(total_extracts_dir, 'MQ_prenylation_variants_stat_te.csv'), index=False)
prenylation_act_te.to_csv(os.path.join(total_extracts_dir, 'MQ_prenylation_variants_act_te.csv'), index=False)
prenylation_FTI_te.to_csv(os.path.join(total_extracts_dir, 'MQ_prenylation_variants_FTI_te.csv'), index=False)

prenylation_act_clickit.to_csv(os.path.join(prenylation_enrichment_dir, 'MQ_prenylation_variants_act_clickit.csv'), index=False)        